# Phase 4 Benchmark: Sort / Limit / Distinct

> Benchmark for Phase 4 MXFrame operations using Mojo sort_indices and unique_mask kernels.

This notebook benchmarks Sort, Limit, and Distinct on synthetic grouped data.
Comparisons: MXFrame CPU vs Polars vs Pandas.

In [ ]:
#| hide
#| eval: false

import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
from max import driver as _driver

from mxframe.lazy_frame import LazyFrame, Scan
from mxframe.lazy_expr import col, lit
from mxframe.custom_ops import clear_cache

try:
    import polars as pl
    POLARS_AVAILABLE = True
except ImportError:
    POLARS_AVAILABLE = False
    print('polars not available — skipping Polars baselines')


## 1) Synthetic data

In [ ]:
#| hide
#| eval: false

def make_data(n: int = 500_000, n_groups: int = 1000, seed: int = 42) -> pa.Table:
    rng = np.random.default_rng(seed)
    groups = np.array([f'g{i:04d}' for i in range(n_groups)], dtype=object)
    cat = groups[rng.integers(0, n_groups, size=n)]
    vals = rng.standard_normal(n).astype(np.float64)
    return pa.table({'group': cat, 'value': vals})

N = 500_000
data = make_data(N)                            # 1000 groups — CPU + Polars
data_gpu = make_data(N, n_groups=50)           # 50 groups — fits GPU 64-group limit
print(f'CPU data: {len(data):,} rows, {len(pc.unique(data.column("group")))} groups')
print(f'GPU data: {len(data_gpu):,} rows, {len(pc.unique(data_gpu.column("group")))} groups')


CPU data: 500,000 rows, 1000 groups
GPU data: 500,000 rows, 50 groups


## 2) Query implementations

In [ ]:
#| hide
#| eval: false

# ── MXFrame ─────────────────────────────────────────────
def mx_sort(tbl, device='cpu'):
    """GroupBy + Agg + Sort by group."""
    return (
        LazyFrame(Scan(tbl))
        .groupby('group')
        .agg(col('value').sum().alias('total'))
        .sort('group')
        .compute(device=device)
    )

def mx_sort_limit(tbl, n=10, device='cpu'):
    """GroupBy + Agg + Sort + Limit."""
    return (
        LazyFrame(Scan(tbl))
        .groupby('group')
        .agg(col('value').sum().alias('total'))
        .sort('group')
        .limit(n)
        .compute(device=device)
    )

def mx_distinct(tbl, device='cpu'):
    """Distinct groups from raw column."""
    return (
        LazyFrame(Scan(tbl))
        .distinct('group')
        .compute(device=device)
    )

# ── Polars ──────────────────────────────────────────────
def pl_sort(tbl):
    if not POLARS_AVAILABLE: return None
    return (
        pl.from_arrow(tbl)
        .group_by('group').agg(pl.col('value').sum().alias('total'))
        .sort('group')
    )

def pl_sort_limit(tbl, n=10):
    if not POLARS_AVAILABLE: return None
    return (
        pl.from_arrow(tbl)
        .group_by('group').agg(pl.col('value').sum().alias('total'))
        .sort('group').head(n)
    )

def pl_distinct(tbl):
    if not POLARS_AVAILABLE: return None
    return pl.from_arrow(tbl).select('group').unique()

# ── Pandas ──────────────────────────────────────────────
def pd_sort(tbl):
    df = tbl.to_pandas()
    return df.groupby('group', as_index=False)['value'].sum().sort_values('group').reset_index(drop=True)

def pd_sort_limit(tbl, n=10):
    return pd_sort(tbl).head(n)

def pd_distinct(tbl):
    return tbl.to_pandas()['group'].drop_duplicates().reset_index(drop=True)

# Quick preview
print('MXFrame Sort preview (first 5):')
print(mx_sort(data).to_pandas().head())

MXFrame Sort preview (first 5):
   group      total
0  g0000 -20.826462
1  g0001  -9.836839
2  g0002 -16.896812
3  g0003 -27.338821
4  g0004  10.359162


## 3) Benchmark

In [ ]:
#| hide
#| eval: false

COLD_RUNS = 3
HOT_RUNS  = 5

def _bench(fn, label, cold=COLD_RUNS, hot=HOT_RUNS, clear=False):
    """Return (label, cold_ms, hot_ms)."""
    times_c = []
    for _ in range(cold):
        if clear: clear_cache()
        t0 = time.perf_counter()
        fn()
        times_c.append((time.perf_counter() - t0) * 1000)
    times_h = []
    for _ in range(hot):
        t0 = time.perf_counter()
        fn()
        times_h.append((time.perf_counter() - t0) * 1000)
    return (label, f'{np.median(times_c):.1f}', f'{np.median(times_h):.1f}')

def _print_table(title, rows):
    hdr = ('', 'Cold ms', 'Hot ms')
    widths = [max(len(str(r[i])) for r in [hdr]+rows) for i in range(3)]
    fmt = '  '.join(f'{{:<{w}}}' for w in widths)
    print(f'\n{title}')
    print(fmt.format(*hdr))
    print('-' * sum(widths + [4]))
    for r in rows:
        print(fmt.format(*r))

# ── GPU readiness ────────────────────────────────────────────────────────
# GPU groupby is limited to 64 groups/launch, so GPU rows use data_gpu (50 groups).
# CPU/Polars/Pandas rows use data (1000 groups).
GPU_READY = False
if _driver.accelerator_count() > 0:
    try:
        _ = mx_sort(data_gpu, device='gpu')
        GPU_READY = True
        print(f'GPU available ({_driver.accelerator_count()} device(s))')
        print('Note: GPU benchmarks use 50-group dataset (GPU kernel limit: 64 groups/launch)')
    except Exception as e:
        print(f'GPU not usable: {e}')
else:
    print('No GPU detected — CPU + Polars + Pandas only')

# ── Sort ────────────────────────────────────────────────
rows_sort = [
    _bench(lambda: mx_sort(data), 'MXFrame CPU (1000g)', clear=True),
    _bench(lambda: pd_sort(data), 'Pandas (1000g)'),
]
if GPU_READY:
    rows_sort.append(_bench(lambda: mx_sort(data_gpu, device='gpu'), 'MXFrame GPU (50g)', clear=True))
    rows_sort.append(_bench(lambda: pd_sort(data_gpu), 'Pandas (50g)'))
if POLARS_AVAILABLE:
    rows_sort.append(_bench(lambda: pl_sort(data), 'Polars (1000g)'))
_print_table('Sort (groupby+agg+sort)', rows_sort)

# ── Sort + Limit ────────────────────────────────────────
rows_sl = [
    _bench(lambda: mx_sort_limit(data), 'MXFrame CPU (1000g)', clear=True),
    _bench(lambda: pd_sort_limit(data), 'Pandas (1000g)'),
]
if GPU_READY:
    rows_sl.append(_bench(lambda: mx_sort_limit(data_gpu, device='gpu'), 'MXFrame GPU (50g)', clear=True))
    rows_sl.append(_bench(lambda: pd_sort_limit(data_gpu), 'Pandas (50g)'))
if POLARS_AVAILABLE:
    rows_sl.append(_bench(lambda: pl_sort_limit(data), 'Polars (1000g)'))
_print_table('Sort + Limit(10)', rows_sl)

# ── Distinct — no aggregation so any N works on GPU ──────
rows_dist = [
    _bench(lambda: mx_distinct(data), 'MXFrame CPU (1000g)', clear=True),
    _bench(lambda: pd_distinct(data), 'Pandas (1000g)'),
]
if GPU_READY:
    rows_dist.append(_bench(lambda: mx_distinct(data, device='gpu'), 'MXFrame GPU (1000g)', clear=True))
if POLARS_AVAILABLE:
    rows_dist.append(_bench(lambda: pl_distinct(data), 'Polars (1000g)'))
_print_table('Distinct (unique groups) — GPU skips aggregation so uses full 1000g', rows_dist)


GPU available (2 device(s))
Note: GPU benchmarks use 50-group dataset (GPU kernel limit: 64 groups/launch)

Sort (groupby+agg+sort)
                     Cold ms  Hot ms
------------------------------------
MXFrame CPU (1000g)  3541.9   6.0   
Pandas (1000g)       17.1     14.2  
MXFrame GPU (50g)    7103.0   10.4  
Pandas (50g)         13.2     12.7  
Polars (1000g)       25.6     23.2  

Sort + Limit(10)
                     Cold ms  Hot ms
------------------------------------
MXFrame CPU (1000g)  3357.2   5.1   
Pandas (1000g)       14.7     14.7  
MXFrame GPU (50g)    6858.9   10.4  
Pandas (50g)         12.7     12.3  
Polars (1000g)       28.1     22.7  

Distinct (unique groups) — GPU skips aggregation so uses full 1000g
                     Cold ms  Hot ms
------------------------------------
MXFrame CPU (1000g)  41.3     37.7  
Pandas (1000g)       12.5     12.4  
MXFrame GPU (1000g)  7195.9   60.9  
Polars (1000g)       12.8     12.7  
